In [1]:
!pip install playwright pandas nest_asyncio lxml
!playwright install chromium

In [10]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import pandas as pd
from playwright.async_api import async_playwright
from io import StringIO

async def brute_force_pai_scraper():
    target_districts = ["Bathinda", "Patiala", "Fatehgarh Sahib", "Rupnagar"] 
    all_data = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={'width': 1280, 'height': 800},
            locale='en-US',
            extra_http_headers={'Accept-Language': 'en-US,en;q=0.9'}
        )
        page = await context.new_page()
        
        print("1. Navigating to PAI portal...")
        await page.goto("https://pai.gov.in/PS/Public/TW-GP.aspx", timeout=90000)
        
        try:
            hindi_indicator = page.locator("text='Hindi (हिन्दी)'")
            if await hindi_indicator.count() > 0:
                print("-> Site loaded in Hindi. Switching to English...")
                await hindi_indicator.first.click()
                await page.wait_for_timeout(1000) 
                english_option = page.locator("text='English'")
                if await english_option.count() > 0:
                    await english_option.first.click()
                    await page.wait_for_timeout(5000) 
        except Exception:
            pass

        print("2. Scanning all dropdowns for State data...")
        try:
            await page.wait_for_function('''
                Array.from(document.querySelectorAll("select")).some(s => 
                    Array.from(s.options).some(o => o.text.toLowerCase().includes("punjab"))
                )
            ''', timeout=60000)
        except Exception:
            print("\nCRITICAL ERROR: 'Punjab' never loaded. Server might be down.")
            await browser.close()
            return None

        print("3. Executing JavaScript to select Punjab...")
        state_result = await page.evaluate('''() => {
            let selects = document.querySelectorAll("select");
            for(let s of selects) {
                for(let i=0; i<s.options.length; i++){
                    let txt = s.options[i].text;
                    if(txt.toLowerCase().includes("punjab")){
                        s.selectedIndex = i;
                        s.dispatchEvent(new Event("change", {bubbles: true}));
                        return { success: true, debug: txt, index: Array.from(selects).indexOf(s) };
                    }
                }
            }
            return { success: false };
        }''')

        if not state_result['success']:
            print(f"\nCRITICAL ERROR: Failed to click Punjab.")
            await browser.close()
            return None

        for dist_name in target_districts:
            print(f"\n--- Loading District: {dist_name} ---")
            
            print(f"  Waiting for {dist_name} data to arrive...")
            try:
                await page.wait_for_function(f'''
                    Array.from(document.querySelectorAll("select")).some(s => 
                        Array.from(s.options).some(o => o.text.toLowerCase().includes("{dist_name.lower()}"))
                    )
                ''', timeout=60000)
            except Exception:
                print(f"  Skipping {dist_name} - Data never arrived.")
                continue
            
            dist_success = await page.evaluate(f'''(districtName) => {{
                let selects = document.querySelectorAll("select");
                for(let j=0; j<selects.length; j++){{
                    let s = selects[j];
                    for(let i=0; i<s.options.length; i++){{
                        if(s.options[i].text.toLowerCase().includes(districtName.toLowerCase())){{
                            s.selectedIndex = i;
                            s.dispatchEvent(new Event("change", {{bubbles: true}}));
                            return true;
                        }}
                    }}
                }}
                return false;
            }}''', dist_name)

            if not dist_success:
                print(f"Skipping {dist_name} - Could not click it.")
                continue

            print(f"  Waiting for blocks to load...")
            await page.wait_for_timeout(5000)

            # Grab all block names safely
            blocks = await page.evaluate(f'''(districtName) => {{
                let selects = document.querySelectorAll("select");
                let distIndex = -1;
                
                // Find where the district dropdown is right now
                for(let j=0; j<selects.length; j++){{
                    let s = selects[j];
                    if(s.options[s.selectedIndex] && s.options[s.selectedIndex].text.toLowerCase().includes(districtName.toLowerCase())){{
                        distIndex = j;
                        break;
                    }}
                }}
                
                if(distIndex === -1 || distIndex + 1 >= selects.length) return [];
                
                let blockDrop = selects[distIndex + 1];
                let options = [];
                for(let i=1; i<blockDrop.options.length; i++){{
                    options.push(blockDrop.options[i].text);
                }}
                return options;
            }}''', dist_name)

            if not blocks:
                print(f"  No blocks found for {dist_name}. Skipping.")
                continue

            print(f"Found {len(blocks)} blocks. Starting extraction...")

            for block_label in blocks:
                print(f"  -> Scraping Block: {block_label}")
                
                # FIXED: Hunt for the specific block dropdown every single loop
                clicked_block = await page.evaluate(f'''(bName) => {{
                    let selects = document.querySelectorAll("select");
                    for(let s of selects) {{
                        for(let i=0; i<s.options.length; i++){{
                            if(s.options[i].text === bName){{
                                s.selectedIndex = i;
                                s.dispatchEvent(new Event("change", {{bubbles: true}}));
                                return true;
                            }}
                        }}
                    }}
                    return false;
                }}''', block_label)
                
                if not clicked_block:
                    print("     [ERROR] Could not find this block in the dropdowns. The page might have refreshed incorrectly.")
                    continue

                await page.wait_for_timeout(2000) 

                await page.evaluate('''() => {
                    let buttons = Array.from(document.querySelectorAll('button, input'));
                    let searchBtn = buttons.find(b => (b.innerText && b.innerText.includes("Search")) || (b.value && b.value.includes("Search")));
                    if(searchBtn) searchBtn.click();
                }''')

                print("     Waiting for the table data to appear...")
                try:
                    # FIXED: Ensure the table has actual data cells (td) loaded before proceeding
                    await page.wait_for_selector("table td", timeout=45000)
                    await page.wait_for_timeout(3000) # Give pandas an extra 3 seconds to ensure DOM is settled
                except Exception:
                    print("     [WARNING] The data table did not load in time.")
                    continue

                try:
                    page_content = await page.content()
                    # Using match to force pandas to only read tables with 'Gram Panchayat Details' in the raw HTML
                    tables = pd.read_html(StringIO(page_content), match='Gram Panchayat Details')
                    
                    extracted = False
                    if tables:
                        for df in tables:
                            # Drop any completely empty columns or rows that might be formatting artifacts
                            df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
                            
                            # FIXED: Ensure we are only saving tables that actually have data rows
                            if not df.empty and len(df) > 0:
                                df['District_Extracted'] = dist_name
                                df['Block_Extracted'] = block_label
                                all_data.append(df)
                                print(f"     [SUCCESS] Saved {len(df)} rows.")
                                extracted = True
                                break
                    
                    if not extracted:
                        print("     [WARNING] Could not find the specific data inside the table.")
                
                except ValueError:
                    print("     [WARNING] Pandas could not find a table matching 'Gram Panchayat Details'.")
                except Exception as e:
                    print(f"     [ERROR] Issue reading HTML table: {e}")

        print("\nAll tasks finished. Closing browser...")
        await browser.close()
        
    if all_data:
        print("\nMerging datasets...")
        final_df = pd.concat(all_data, ignore_index=True)
        csv_filename = "PAI_Village_Data_Complete.csv"
        final_df.to_csv(csv_filename, index=False)
        print(f"100% COMPLETE! Your data is ready in: {csv_filename}")
        return final_df
    else:
        print("\nScraper finished but no data was successfully collected.")
        return None

final_dataset = asyncio.run(brute_force_pai_scraper())

if final_dataset is not None:
    display(final_dataset.head())

1. Navigating to PAI portal...
-> Site loaded in Hindi. Switching to English...
2. Scanning all dropdowns for State data...
3. Executing JavaScript to select Punjab...

--- Loading District: Bathinda ---
  Waiting for Bathinda data to arrive...
  Waiting for blocks to load...
Found 9 blocks. Starting extraction...
  -> Scraping Block: Bathinda [226]
     Waiting for the table data to appear...
     [SUCCESS] Saved 32 rows.
  -> Scraping Block: Bhagta Bhaika [227]
     Waiting for the table data to appear...
     [SUCCESS] Saved 29 rows.
  -> Scraping Block: Goniana [7333]
     Waiting for the table data to appear...
     [SUCCESS] Saved 37 rows.
  -> Scraping Block: Maur [228]
     Waiting for the table data to appear...
     [SUCCESS] Saved 32 rows.
  -> Scraping Block: Nathana [229]
     Waiting for the table data to appear...
     [SUCCESS] Saved 36 rows.
  -> Scraping Block: Phul [230]
     Waiting for the table data to appear...
     [SUCCESS] Saved 25 rows.
  -> Scraping Block: R

,Gram Panchayat Details,Overall PAI Score,T1 - Poverty Free and Enhanced Livelihoods Panchayat,T2 - Healthy Panchayat,T3 - Child Friendly Panchayat,T4 - Water Sufficient Panchayat,T5 - Clean and Green Panchayat,T6 - Self-sufficient Infrastructure in Panchayat,T7 - Socially Just and Socially Secured Panchayat,T8 - Panchayat with Good Governance,District_Extracted,Block_Extracted,T9 - Women Friendly Panchayat
0,GP - Teona-[10719] Block - Bathinda District -...,66.82 B Performer,79.57 A Front Runner,80.59 A Front Runner,75.28 A Front Runner,57.84 C Aspirant,49.87 C Aspirant,65.6 B Performer,73.8 B Performer,70.71 B Performer,Bathinda,Bathinda [226],NaN
1,GP - Virk Khurd-[10721] Block - Bathinda Distr...,66.72 B Performer,73.95 B Performer,87.86 A Front Runner,72.41 B Performer,61.8 B Performer,50.47 C Aspirant,60.63 B Performer,82.19 A Front Runner,63.93 B Performer,Bathinda,Bathinda [226],NaN
2,GP - Virk Kalan-[10720] Block - Bathinda Distr...,66.59 B Performer,77.13 A Front Runner,74.65 B Performer,75.78 A Front Runner,72.13 B Performer,50.01 C Aspirant,54.5 C Aspirant,80.53 A Front Runner,65.88 B Performer,Bathinda,Bathinda [226],NaN
3,GP - Naruana-[10713] Block - Bathinda District...,66.15 B Performer,75.01 A Front Runner,79.53 A Front Runner,72.06 B Performer,59.97 C Aspirant,57.84 C Aspirant,61.98 B Performer,78.28 A Front Runner,66.49 B Performer,Bathinda,Bathinda [226],NaN
4,GP - Jhumba-[10698] Block - Bathinda District ...,65.89 B Performer,80.84 A Front Runner,80.71 A Front Runner,69 B Performer,68.63 B Performer,49.15 C Aspirant,56.02 C Aspirant,75.35 A Front Runner,72.08 B Performer,Bathinda,Bathinda [226],NaN


In [3]:
import pandas as pd
import numpy as np
import re

print("Loading raw dataset...")
df = pd.read_csv("PAI_Village_Data_Complete.csv")

# 1. Standardize original columns to make searching easier
df.columns = df.columns.str.strip().str.replace('\n', ' ').str.replace('  ', ' ')

# 2. Advanced Extraction Function: Grabs Score, Grade, AND Category
def extract_all_components(val):
    if pd.isna(val):
        return pd.Series([np.nan, 'Unknown', 'Unknown'])
    
    val_str = str(val).strip()
    match = re.search(r'([\d\.]+)\s+([A-D][+]?)\s+(.*)', val_str)
    
    if match:
        score = float(match.group(1))
        grade = match.group(2).strip()
        category = match.group(3).strip()
        return pd.Series([score, grade, category])
    
    num_match = re.search(r'\d+(\.\d+)?', val_str)
    score = float(num_match.group(0)) if num_match else np.nan
    
    grade_match = re.search(r'\b([A-D][+]?)\b', val_str)
    grade = grade_match.group(1) if grade_match else "Unknown"
    
    return pd.Series([score, grade, "Unknown"])

# 3. Create the Base DataFrame with ID mapping (THE NEW FIX)
print("Extracting GP names and LGD IDs...")
final_df = pd.DataFrame()

# Find the column regardless of hidden sort arrows
gp_col_matches = [c for c in df.columns if 'Gram Panchayat' in c]

if gp_col_matches:
    gp_col = gp_col_matches[0]
    
    # Extract GP_ID: Find any digits inside square brackets
    final_df['GP_ID'] = df[gp_col].str.extract(r'\[(\d+)\]')[0]
    
    # Extract GP_Name: Look for the text strictly between "GP - " and "-["
    final_df['GP_Name'] = df[gp_col].str.extract(r'GP\s*-\s*(.+?)-\[')[0].str.strip()
    
else:
    print("Warning: Could not find any column containing 'Gram Panchayat'.")

final_df['Block'] = df.get('Block_Extracted', 'Unknown')
final_df['District'] = df.get('District_Extracted', 'Unknown')
final_df['State'] = 'Punjab'

# 4. Map the messy raw columns to your desired prefixes
prefix_mapping = {
    'Overall PAI Score': 'Overall_PAI',
    'T1 - Poverty Free': 'T1_Poverty_Free',
    'T2 - Healthy': 'T2_Healthy',
    'T3 - Child Friendly': 'T3_Child_Friendly',
    'T4 - Water Sufficient': 'T4_Water_Sufficient',
    'T5 - Clean and Green': 'T5_Clean_Green',
    'T6 - Self-sufficient': 'T6_Self_Sufficient',
    'T7 - Socially Just': 'T7_Socially_Just',
    'T8 - Panchayat with Good Governance': 'T8_Good_Governance',
    'T9 - Women Friendly': 'T9_Women_Friendly'
}

print("Parsing metrics into Score, Grade, and Category columns...")

for partial_raw_name, clean_prefix in prefix_mapping.items():
    matching_cols = [c for c in df.columns if partial_raw_name in c]
    
    if matching_cols:
        raw_col = matching_cols[0] 
        extracted_data = df[raw_col].apply(extract_all_components)
        
        if clean_prefix == 'Overall_PAI':
            final_df['Overall_PAI_Score'] = extracted_data[0]
            final_df['Overall_Grade'] = extracted_data[1]
            final_df['Overall_Category'] = extracted_data[2]
        else:
            final_df[f'{clean_prefix}_Score'] = extracted_data[0]
            final_df[f'{clean_prefix}_Grade'] = extracted_data[1]
            final_df[f'{clean_prefix}_Category'] = extracted_data[2]

# 5. Smart Missing Data Imputation
print("Handling missing data...")
score_cols = [col for col in final_df.columns if col.endswith('_Score')]

for col in score_cols:
    final_df[col] = final_df.groupby('Block')[col].transform(lambda x: x.fillna(x.median()))
    final_df[col] = final_df.groupby('District')[col].transform(lambda x: x.fillna(x.median()))

# 6. Final cleanup and saving
cleaned_filename = "PAI_Village_Data_Machine_Ready.csv"
final_df.to_csv(cleaned_filename, index=False)

print(f"\nProcessing Complete! Dataset formatted perfectly and saved to: {cleaned_filename}")

# Display the final structure to verify
display(final_df.head())

Loading raw dataset...
Extracting GP names and LGD IDs...
Parsing metrics into Score, Grade, and Category columns...
Handling missing data...

Processing Complete! Dataset formatted perfectly and saved to: PAI_Village_Data_Machine_Ready.csv


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,GP_ID,GP_Name,Block,District,State,Overall_PAI_Score,Overall_Grade,Overall_Category,T1_Poverty_Free_Score,T1_Poverty_Free_Grade,...,T6_Self_Sufficient_Category,T7_Socially_Just_Score,T7_Socially_Just_Grade,T7_Socially_Just_Category,T8_Good_Governance_Score,T8_Good_Governance_Grade,T8_Good_Governance_Category,T9_Women_Friendly_Score,T9_Women_Friendly_Grade,T9_Women_Friendly_Category
0,10719,Teona,Bathinda [226],Bathinda,Punjab,66.82,B,Performer,79.57,A,...,Performer,73.80,B,Performer,70.71,B,Performer,52.63,Unknown,Unknown
1,10721,Virk Khurd,Bathinda [226],Bathinda,Punjab,66.72,B,Performer,73.95,B,...,Performer,82.19,A,Front Runner,63.93,B,Performer,52.63,Unknown,Unknown
2,10720,Virk Kalan,Bathinda [226],Bathinda,Punjab,66.59,B,Performer,77.13,A,...,Aspirant,80.53,A,Front Runner,65.88,B,Performer,52.63,Unknown,Unknown
3,10713,Naruana,Bathinda [226],Bathinda,Punjab,66.15,B,Performer,75.01,A,...,Performer,78.28,A,Front Runner,66.49,B,Performer,52.63,Unknown,Unknown
4,10698,Jhumba,Bathinda [226],Bathinda,Punjab,65.89,B,Performer,80.84,A,...,Aspirant,75.35,A,Front Runner,72.08,B,Performer,52.63,Unknown,Unknown
